# Cas d'étude : De 2 600 lignes à 4 graphiques en 40 minutes

Ce notebook accompagne le cas d'étude sur l'analyse de données de consommation énergétique.

**Objectif** : explorer ~2 600 relevés, identifier et nettoyer les anomalies, puis générer 4 graphiques pour communiquer les résultats.

**Données** : `export_gtc_brut.csv` contient les colonnes suivantes :
- `date` : date du relevé
- `batiment` : Usine Nord ou Usine Sud
- `zone` : Production, Logistique, Tertiaire ou Maintenance
- `kwh` : consommation énergétique
- `temperature_ext` : température extérieure
- `occupation_pct` : taux d'occupation
- `type_energie` : electricite ou gaz


## Phase 1 : Exploration et diagnostic

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Charger les données brutes
df_brut = pd.read_csv("../../data/export_gtc_brut.csv", parse_dates=["date"])

print("Aperçu des données :")
print(df_brut.head())
print(f"\nDimensions : {df_brut.shape}")

In [ ]:
# Diagnostic 1 : valeurs manquantes
print("=== Valeurs manquantes ===")
missing = df_brut.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante")
print(f"Total : {missing.sum()}")

In [ ]:
# Diagnostic 2 : valeurs négatives dans kwh (anomalie métier)
print("=== Consommations négatives ===")
negative_kwh = (df_brut['kwh'] < 0).sum()
print(f"Nombre de lignes avec kwh < 0 : {negative_kwh}")
if negative_kwh > 0:
    print("\nExemples :")
    print(df_brut[df_brut['kwh'] < 0][['date', 'batiment', 'zone', 'kwh']].head())

In [ ]:
# Diagnostic 3 : doublons et format de date
print("=== Doublons ===")
duplicates = df_brut.duplicated().sum()
print(f"Nombre de lignes dupliquées : {duplicates}")

print("\n=== Format de la date ===")
print(f"Type : {df_brut['date'].dtype}")
print(f"Plage : {df_brut['date'].min()} à {df_brut['date'].max()}")

### Résumé du diagnostic

**20 anomalies détectées :**
- Valeurs manquantes dans `kwh` : contribuent aux imputations
- Consommations négatives : impossibles métier
- Doublons : lignes identiques

Ces anomalies vont être nettoyées en phase 2.

## Phase 2 : Nettoyage en 3 étapes

In [ ]:
# Travailler sur une copie
df = df_brut.copy()

print(f"État initial : {len(df)} lignes, {df.isnull().sum().sum() + (df['kwh'] < 0).sum() + df.duplicated().sum()} anomalies")

# Étape 1 : Supprimer les doublons
df = df.drop_duplicates()
print(f"Après drop_duplicates() : {len(df)} lignes")

# Étape 2 : Filtrer les valeurs négatives
df = df[df['kwh'] >= 0]
print(f"Après filtrage kwh >= 0 : {len(df)} lignes")

In [ ]:
# Étape 3 : Imputer les valeurs manquantes par la moyenne par groupe
# Grouper par bâtiment, zone et type d'énergie, puis remplir avec la moyenne du groupe
df['kwh'] = df.groupby(['batiment', 'zone', 'type_energie'])['kwh'].transform(
    lambda x: x.fillna(x.mean())
)

print(f"Après imputation : {len(df)} lignes")
print(f"Valeurs manquantes restantes : {df.isnull().sum().sum()}")

In [ ]:
# Sauvegarder le dataset nettoyé
df.to_csv("../../data/export_gtc_propre.csv", index=False)

print("\n=== Comparaison avant/après ===")
print(f"Brut        : {len(df_brut)} lignes")
print(f"Propre      : {len(df)} lignes")
print(f"Différence  : {len(df_brut) - len(df)} lignes supprimées")
print(f"\nAnomalies brut  : ~22")
print(f"Anomalies propre : 0")

## Phase 3 : Les 4 graphiques

### Graphique 1 : Évolution mensuelle par bâtiment

In [ ]:
# Créer colonne mois-année pour l'agrégation
df_g1 = df.copy()
df_g1['mois'] = df_g1['date'].dt.to_period('M')

# Agréger par mois et bâtiment
consumption_by_month = df_g1.groupby(['mois', 'batiment'])['kwh'].sum().reset_index()
consumption_by_month['mois'] = consumption_by_month['mois'].astype(str)

# Créer le graphique ligne
fig1 = px.line(
    consumption_by_month,
    x='mois',
    y='kwh',
    color='batiment',
    markers=True,
    title='Consommation énergétique mensuelle par bâtiment',
    labels={'kwh': 'Consommation (kWh)', 'mois': 'Mois'}
)

fig1.show()fig1.write_html('../../output/graphique_1_evolution.html')


### Graphique 2 : Nuage de points gaz et électricité vs température


In [ ]:
# Scatter plot avec les deux types d'énergie et une ligne de tendance par type
fig2 = px.scatter(
    df,
    x='temperature_ext',
    y='kwh',
    color='type_energie',
    trendline='ols',
    title='Consommation vs température — gaz et électricité',
    labels={'temperature_ext': 'Température extérieure (°C)', 'kwh': 'Consommation (kWh)', 'type_energie': 'Type'}
)
fig2.show()
fig2.write_html('../../output/graphique_2_correlation.html')

# Coefficient de corrélation pour chaque type d'énergie
print('=== Coefficient de corrélation température / consommation ===')
for energie in df['type_energie'].unique():
    corr = df[df['type_energie'] == energie]['temperature_ext'].corr(df[df['type_energie'] == energie]['kwh'])
    print(f'{energie:12s} : r = {corr:.2f}')


### Graphique 3 : Heatmap zone × semaine

In [ ]:
# Créer colonne numéro de semaine
df_g3 = df.copy()
df_g3['semaine'] = df_g3['date'].dt.isocalendar().week

# Pivot table : zone en ligne, semaine en colonne
heatmap_data = df_g3.groupby(['zone', 'semaine'])['kwh'].sum().reset_index()
pivot_heat = heatmap_data.pivot(index='zone', columns='semaine', values='kwh')

# Créer la heatmap
fig3 = px.imshow(
    pivot_heat,
    color_continuous_scale='RdBu_r',
    title='Consommation par zone et semaine',
    labels={'zone': 'Zone', 'semaine': 'Numéro de semaine', 'color': 'kWh'}
)

fig3.show()
fig3.write_html('../../output/graphique_3_heatmap.html')


### Graphique 4 : Dashboard 2×2 pour impression (PDF/PNG)

In [ ]:
# Préparer les données pour chaque sous-graphique

# Sous-graphique 1 : consommation totale par type d'énergie
energy_by_type = df.groupby('type_energie')['kwh'].sum()

# Sous-graphique 2 : consommation par zone
energy_by_zone = df.groupby('zone')['kwh'].sum().sort_values(ascending=False)

# Sous-graphique 3 : occupation vs consommation électrique
df_elec = df[df['type_energie'] == 'electricite'].copy()
occupancy_vs_consumption = df_elec.groupby(pd.cut(df_elec['occupation_pct'], bins=5)).agg({'kwh': 'mean'})

# Sous-graphique 4 : évolution hebdomadaire
df_weekly = df.copy()
df_weekly['semaine'] = df_weekly['date'].dt.isocalendar().week
weekly_consumption = df_weekly.groupby('semaine')['kwh'].sum()

# Créer la figure matplotlib 2×2, A4 landscape
fig, axes = plt.subplots(2, 2, figsize=(11.69, 8.27), dpi=300)
fig.suptitle('Analyse énergétique du bâtiment', fontsize=16, fontweight='bold')

# (1,1) Consommation par type d'énergie
axes[0, 0].bar(energy_by_type.index, energy_by_type.values, color=['#1f77b4', '#ff7f0e'])
axes[0, 0].set_title('Consommation par type d\'énergie')
axes[0, 0].set_ylabel('kWh')
axes[0, 0].grid(axis='y', alpha=0.3)

# (1,2) Consommation par zone
axes[0, 1].barh(energy_by_zone.index, energy_by_zone.values, color='#2ca02c')
axes[0, 1].set_title('Consommation par zone')
axes[0, 1].set_xlabel('kWh')
axes[0, 1].grid(axis='x', alpha=0.3)

# (2,1) Occupation vs consommation électrique
axes[1, 0].plot(range(len(occupancy_vs_consumption)), occupancy_vs_consumption['kwh'].values, marker='o', color='#d62728')
axes[1, 0].set_title('Consommation électrique vs occupation')
axes[1, 0].set_ylabel('kWh moyen')
axes[1, 0].set_xlabel('Taux d\'occupation')
axes[1, 0].grid(alpha=0.3)

# (2,2) Évolution hebdomadaire
axes[1, 1].plot(weekly_consumption.index, weekly_consumption.values, marker='s', color='#9467bd', linewidth=2)
axes[1, 1].fill_between(weekly_consumption.index, weekly_consumption.values, alpha=0.3, color='#9467bd')
axes[1, 1].set_title('Consommation hebdomadaire')
axes[1, 1].set_xlabel('Semaine')
axes[1, 1].set_ylabel('kWh')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()

# Sauvegarder en PDF et PNG
plt.savefig('../../output/dashboard_direction.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.savefig('../../output/dashboard_direction.png', format='png', dpi=300, bbox_inches='tight')

print("Dashboard sauvegardé en PDF et PNG")
plt.show()

## Phase 4 : Insights métier

### Les 3 insights clés

**1. Le chauffage est le principal levier d'action**  
Les deux bâtiments consomment environ 1,8× plus en hiver qu'en été. Les profils sont comparables (Usine Nord ~326 kWh/j, Usine Sud ~338 kWh/j). La saisonnalité est le signal dominant — c'est le chauffage qui pilote la consommation.

**2. Le gaz est piloté par la météo, l'électricité non**  
Corrélation gaz / température : r ≈ −0.83 (forte corrélation inverse). Corrélation électricité / température : r ≈ 0. Ce sont deux problèmes d'optimisation distincts : isolation et pilotage du chauffage pour le gaz ; usages machines et éclairage pour l'électricité.

**3. La zone Production concentre plus de 50 % de la consommation**  
Logistique, Tertiaire et Maintenance se partagent le reste à parts quasi égales. Un point de réduction en Production vaut trois points sur les autres zones combinées.

---

### Excel vs Python + Claude Code

| Étape | Excel | Python + Claude Code |
|-------|-------|---------------------|
| Détection des doublons | Mise en forme conditionnelle, manuelle | 1 ligne de code |
| Remplacement des négatifs | Filtre + saisie manuelle | 2 lignes de code |
| Imputation par groupe | Formule MOYENNE.SI, colonne auxiliaire | 5 lignes de code |
| 4 graphiques | 4 onglets, configuration manuelle | 4 blocs de code |
| Mise à jour si nouvelles données | Tout refaire | Relancer le script |
| **Total estimé** | **~2 heures** | **~40 minutes** |
